# Checkpoint Loader
Load any saved checkpoint and generate text.

In [ ]:
import os
import math
from typing import Dict, List, Tuple
import re

import torch
import torch.nn as nn

# ---- SET THIS TO YOUR CHECKPOINT ----
# CKPT_PATH = 'checkpoints/best/val_ppl=63.70.pt'
# CKPT_PATH = 'checkpoints/best/val_ppl=.pt'
# CKPT_PATH = 'checkpoints/20260313123822/epoch_00.pt'
CKPT_PATH = 'checkpoints/20260318010037/epoch_002.pt'
# -------------------------------------

DATA_DIR = 'data'
VOCAB_PATH = os.path.join(DATA_DIR, 'vocab.txt')
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

print('DEVICE:', DEVICE)
print('CKPT_PATH:', CKPT_PATH)

DEVICE: cuda
CKPT_PATH: checkpoints/20260318010037/epoch_002.pt


In [ ]:
DIGIT_RE = re.compile(r"\d+")
WHITESPACE_RE = re.compile(r"\s+")

# Optional: normalize curly quotes / dashes to plain forms
CHAR_MAP = str.maketrans({
    "“": '"', "”": '"', "„": '"',
    "’": "'", "‘": "'",
    "—": "-", "–": "-",
    "…": "...",
})

def normalize_text(text: str) -> str:
    # normalize line endings
    text = text.replace("\r\n", "\n").replace("\r", "\n")
    # unify common unicode punctuation
    text = text.translate(CHAR_MAP)
    # lowercase
    text = text.lower()
    # numbers -> <num>
    text = DIGIT_RE.sub(" <num> ", text)
    # collapse whitespace
    text = WHITESPACE_RE.sub(" ", text).strip()
    return text

## Load vocab

In [2]:
def load_vocab(vocab_path: str) -> Tuple[List[str], Dict[str, int]]:
    with open(vocab_path, 'r', encoding='utf-8') as f:
        vocab = [line.strip() for line in f if line.strip()]
    tok2id = {t: i for i, t in enumerate(vocab)}
    return vocab, tok2id

vocab, tok2id = load_vocab(VOCAB_PATH)
pad_id = tok2id.get('<pad>', 0)
bos_id = tok2id.get('<bos>')
eos_id = tok2id.get('<eos>')

print('vocab size:', len(vocab))

vocab size: 30000


## Model

In [3]:
class LSTMLM(nn.Module):
    def __init__(self, vocab_size: int, emb_dim: int, hidden: int, num_layers: int, dropout: float, pad_id: int):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, emb_dim, padding_idx=pad_id)
        self.lstm = nn.LSTM(
            input_size=emb_dim,
            hidden_size=hidden,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0,
        )
        self.drop = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden, vocab_size)

    def forward(self, x):
        e = self.emb(x)
        out, _ = self.lstm(e)
        out = self.drop(out)
        logits = self.fc(out)
        return logits

## Load checkpoint

In [4]:
ckpt = torch.load(CKPT_PATH, map_location='cpu', weights_only=False)

hp = ckpt['hyperparams']

print('=== Checkpoint info ===')
print(f"  path:      {CKPT_PATH}")
print(f"  epoch:     {ckpt.get('epoch', '?')}")
print(f"  val_loss:  {ckpt.get('val_loss', '?')}")
print(f"  val_ppl:   {ckpt.get('val_ppl', '?')}")
if 'train_loss' in ckpt:
    print(f"  train_loss:{ckpt['train_loss']}")
if 'global_step' in ckpt:
    print(f"  step:      {ckpt['global_step']}")
print()
print('=== Hyperparams ===')
for k, v in hp.items():
    print(f"  {k}: {v}")

# Build model from stored hyperparams
model = LSTMLM(
    vocab_size=hp['vocab_size'],
    emb_dim=hp['emb_dim'],
    hidden=hp['hidden_size'],
    num_layers=hp['num_layers'],
    dropout=hp['dropout'],
    pad_id=pad_id,
).to(DEVICE)

model.load_state_dict(ckpt['model_state_dict'])
model.eval()

total_params = sum(p.numel() for p in model.parameters())
print(f"\nModel loaded: {total_params:,} params")

=== Checkpoint info ===
  path:      checkpoints/20260318010037/epoch_002.pt
  epoch:     2
  val_loss:  4.228449110708761
  val_ppl:   68.61074195296074
  train_loss:3.8943975530047696
  step:      9942

=== Hyperparams ===
  seed: 42
  seq_len: 50
  stride: 10
  batch_size: 256
  grad_clip: 1.0
  emb_dim: 256
  num_layers: 2
  vocab_size: 30000
  lr: 0.00075
  hidden_size: 2048
  dropout: 0.4
  steps_per_epoch: 5000
  max_epochs: 50
  early_stop_patience: 4
  sched_factor: 0.5
  sched_patience: 1
  sched_min_lr: 1e-05
  train_ratio: 0.8
  val_ratio: 0.1
  test_ratio: 0.1

Model loaded: 121,611,568 params


## Sampling utilities

In [ ]:
def encode_prompt(text: str) -> List[int]:
    toks = text.split()
    return [tok2id.get(t, tok2id.get('<unk>', 0)) for t in toks]

def decode(ids: List[int]) -> str:
    toks = [vocab[i] if 0 <= i < len(vocab) else '<unk>' for i in ids]
    text = ' '.join(toks)
    text = (text
            .replace(' ,', ',').replace(' .', '.').replace(' !', '!').replace(' ?', '?')
            .replace(' ;', ';').replace(' :', ':')
            .replace(' )', ')').replace('( ', '(')
            .replace(' - ', '-')
           )
    return text

@torch.no_grad()
def sample_ids(model: nn.Module, prompt_ids: List[int], max_new_tokens: int = 120,
               temperature: float = 1.0, top_k: int = 50) -> List[int]:
    model.eval()
    x = torch.tensor([prompt_ids], dtype=torch.long, device=DEVICE)
    for _ in range(max_new_tokens):
        logits = model(x)[:, -1, :]
        logits = logits / max(temperature, 1e-6)

        if top_k and top_k > 0:
            v, ix = torch.topk(logits, k=min(top_k, logits.size(-1)))
            probs = torch.softmax(v, dim=-1)
            next_local = torch.multinomial(probs, num_samples=1)
            next_id = ix.gather(-1, next_local)
        else:
            probs = torch.softmax(logits, dim=-1)
            next_id = torch.multinomial(probs, num_samples=1)

        next_id_int = int(next_id.item())
        x = torch.cat([x, torch.tensor([[next_id_int]], device=DEVICE)], dim=1)

        if eos_id is not None and next_id_int == eos_id:
            break
        if x.size(1) > 512:
            x = x[:, -512:]

    return x[0].tolist()

def generate(prompt: str, max_new_tokens: int = 120, temperature: float = 0.8, top_k: int = 50):
    ids = encode_prompt(normalize_text(prompt))
    out = sample_ids(model, ids, max_new_tokens=max_new_tokens, temperature=temperature, top_k=top_k)
    text = decode(out)
    print(f'Prompt: {prompt}')
    print(f'---')
    print(text)
    print()

print('Ready. Use generate("your prompt here") to sample text.')

Ready. Use generate("your prompt here") to sample text.


## Generate

In [6]:
generate('once upon a time')

Prompt: once upon a time
---
once upon a time there was a young couple who were married to many others who had travelled among the hills, and they were very happy. they had many children. they loved a little girl with the most romantic and wonderful characters. but a prince, or she, who lived in a large house, had lived many years in a barrio all around. when the king had gone to live with his wife, he was very fond of her. he had a <unk>, who was a lovely boy, and lived in a place where the boy lived and had a daughter. at that time he was very much grieved. he was



In [7]:
generate('the princess')

Prompt: the princess
---
the princess was very glad. " i am very sorry, " she said, " but you must have heard that i was a boy, and i have not heard how much i have been able to make. but we all have been good-for the world will be quite useless. " " i am glad you see, jeanne! " jeanne exclaimed. " it would be an unfortunate task if i am not to bother this strange subject to your poor brother. i will go and get the money for him to take care of it. " " you must not keep on going, " said dorothy. " but



In [8]:
generate('in a dark forest')

Prompt: in a dark forest
---
in a dark forest, and when they had gone up a strange bridge they found a dark forest. it was a beautiful valley. and they went through the woods; it looked like the sunshine and the <unk>. the green trees were so high when they arrived in the spring, they called them and they went and told all the birds. " what are you doing, aponibolinayen, and we shall be married to the great spirit, " she said. " yes, " said the people who were dipping water, " do not be afraid. " so ligi went to the <unk> and told <unk> to get down. as soon as



In [ ]:
generate('once upon a time', max_new_tokens=120, temperature=0.2, top_k=30)

In [ ]:
generate('the princess', max_new_tokens=120, temperature=0.2, top_k=30)

In [ ]:
generate('in a dark forest', max_new_tokens=120, temperature=0.2, top_k=30)